# Initial Questions

1. Our corpus will have 12 documents
2. The average document length is
3. The min length is: and the max length is:
4. The OHCO structure varies based on the novel in the corpus. For The Mysteries of Udolpho, it is ['vol_num', 'chap_num', 'para_num', 'sent_num', 'token_num'] ,for THe Old ENglish Baron, it is ['para_num', 'sent_num', 'token_num'], for the Castle of Otranto, it is ['chap_num', 'para_num', 'sent_num', 'token_num'], for Melmoth the Wanderer, it is ['chap_num', 'para_num', 'sent_num', 'token_num']

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='ticks')

In [2]:
data_home = '../input/datasets/mohiniguptarde/'
output_dir = '../working'

# High Gothic

## The Mysteries of Udolpho

token/vocab

In [3]:
src_file = f"{data_home}/high-gothic/pg3268.txt"


In [4]:
OHCO = ['vol_num', 'chap_num', 'para_num', 'sent_num', 'token_num']

In [5]:
LINES = pd.DataFrame(open(src_file, 'r', encoding='utf-8-sig').readlines(), columns=['line_str'])
LINES.index.name = 'line_num'
LINES.line_str = LINES.line_str.str.replace(r'\n+', ' ', regex=True).str.strip()
title = LINES.loc[0].line_str.replace('The Project Gutenberg EBook of ', '')
clip_pats = [
    r"\*\*\*\s*START OF (?:THE|THIS) PROJECT",
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT"
]
pat_a = LINES.line_str.str.match(clip_pats[0])
pat_b = LINES.line_str.str.match(clip_pats[1])
line_a = LINES.loc[pat_a].index[0] + 95
line_b = LINES.loc[pat_b].index[0] - 1
LINES = LINES.loc[line_a : line_b]
vol_pat = r"^\s*(?:volume)\s+\d+"
chap_pat = r"^\s*(?:chapter)\s+[IVXLCDM]+"
vol_lines = LINES.line_str.str.match(vol_pat, case=False) # Returns a truth vector
#LINES.loc[vol_lines] # Use as filter for dataframe
LINES.loc[vol_lines, 'vol_num'] = [i+1 for i in range(LINES.loc[vol_lines].shape[0])]
LINES.vol_num = LINES.vol_num.ffill()

In [6]:
LINES = LINES.dropna(subset=['vol_num']) # Remove everything before Chapter 1
LINES = LINES.loc[~vol_lines] # Remove chapter heading lines; their work is done
LINES.vol_num = LINES.vol_num.astype('int') # Convert chap_num from float to int

In [7]:
VOLS = LINES.groupby(OHCO[:1])\
    .line_str.apply(lambda x: '\n'.join(x))\
    .to_frame('vol_str')
VOLS['vol_str'] = VOLS.vol_str.str.strip()


In [8]:
#VOLS

In [9]:
chap_lines = LINES.line_str.str.match(chap_pat, case=False) # Returns a truth vector
LINES.loc[chap_lines, 'chap_num'] = (LINES.loc[chap_lines].groupby('vol_num').cumcount() + 1)
LINES['chap_num'] = LINES['chap_num'].ffill()
LINES = LINES.dropna(subset=['chap_num'])
LINES = LINES.loc[~chap_lines]
LINES['chap_num'] = LINES['chap_num'].astype(int)
#LINES.sample(5)

In [10]:
CHAPS = LINES.groupby(OHCO[:2])\
    .line_str.apply(lambda x: '\n'.join(x))\
    .to_frame('chap_str')
CHAPS['chap_str'] = CHAPS.chap_str.str.strip()


In [11]:
para_pat = r'\n\n+'
PARAS = CHAPS['chap_str'].str.split(para_pat, expand=True).stack()\
    .to_frame('para_str').sort_index()
PARAS.index.names = OHCO[:3]

In [12]:
#PARAS.head()

In [13]:
PARAS['para_str'] = PARAS['para_str'].str.replace(r'\n', ' ', regex=True)
PARAS['para_str'] = PARAS['para_str'].str.strip()
PARAS = PARAS[~PARAS['para_str'].str.match(r'^\s*$')] # Remove empty paragraphs

In [14]:
sent_pat = r'[.?!;:]+'
SENTS = PARAS['para_str'].str.split(sent_pat, expand=True).stack()\
    .to_frame('sent_str')
SENTS.index.names = OHCO[:4]
SENTS = SENTS[~SENTS['sent_str'].str.match(r'^\s*$')] # Remove empty paragraphs
SENTS.sent_str = SENTS.sent_str.str.strip() # CRUCIAL TO REMOVE BLANK TOKENS

In [15]:
token_pat = r"[\s',-]+"
TOKENS = SENTS['sent_str'].str.split(token_pat, expand=True).stack()\
    .to_frame('token_str')
TOKENS.index.names = OHCO[:5]
TOKENS
TOKENS['term_str'] = TOKENS.token_str.replace(r'[\W_]+', '', regex=True).str.lower()
VOCAB = TOKENS.term_str.value_counts().to_frame('n').reset_index().rename(columns={'index':'term_str'})
VOCAB.index.name = 'term_id'
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()



In [16]:
TOKENS.head()

token_str term_str
vol_num chap_num para_num sent_num token_num                   
1       1        0        0        0              home     home
                                   1                is       is
                                   2               the      the
                                   3            resort   resort
                                   4                Of       of

In [17]:
VOCAB.head()

,term_str,n,p
term_id,,,
0,the,18998,0.064508
1,and,9776,0.033194
2,to,9643,0.032743
3,of,9515,0.032308
4,her,5690,0.019320


In [18]:
TOKENS = TOKENS[TOKENS.index.get_level_values('vol_num') == 1]
TOKENS = TOKENS.droplevel('vol_num')

In [19]:
udolpho_VOLS = VOLS
udolpho_CHAPS = CHAPS
udolpho_PARAS = PARAS
udolpho_SENTS = SENTS
udolpho_TOKENS = TOKENS
udolpho_VOCAB = VOCAB

In [20]:
udolpho_TOKENS.to_csv(f"{output_dir}/udolpho-TOKENS.csv")


## The Old English Baron: a Gothic Story

In [21]:
src_file = f"{data_home}/high-gothic/pg5182.txt"


In [22]:
OHCO = ['para_num', 'sent_num', 'token_num']

In [23]:
LINES = pd.DataFrame(open(src_file, 'r', encoding='utf-8-sig').readlines(), columns=['line_str'])
LINES.index.name = 'line_num'
LINES.line_str = LINES.line_str.str.replace(r'\n+', ' ', regex=True).str.strip()
title = LINES.loc[0].line_str.replace('The Project Gutenberg EBook of ', '')
clip_pats = [
    r"\*\*\*\s*START OF (?:THE|THIS) PROJECT",
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT"
]
pat_a = LINES.line_str.str.match(clip_pats[0])
pat_b = LINES.line_str.str.match(clip_pats[1])
line_a = LINES.loc[pat_a].index[0] + 1
line_b = LINES.loc[pat_b].index[0] - 1
LINES = LINES.loc[line_a : line_b]

In [24]:
para_pat = r'\n\n+'
PARAS = LINES['line_str'].str.split(para_pat, expand=True).stack().reset_index(drop=True)\
    .to_frame('para_str')
PARAS.index.names = OHCO[:1]

In [25]:
PARAS['para_str'] = PARAS['para_str'].str.replace(r'\n', ' ', regex=True)
PARAS['para_str'] = PARAS['para_str'].str.strip()
PARAS = PARAS[~PARAS['para_str'].str.match(r'^\s*$')] # Remove empty paragraphs

In [26]:
PARAS.head()

,para_str
para_num,
4,Produced by Jack Voller
13,THE OLD ENGLISH BARON
16,By Clara Reeve
21,PREFACE
25,"As this Story is of a species which, though no..."


In [27]:
sent_pat = r'[.?!;:]+'
SENTS = PARAS['para_str'].str.split(sent_pat, expand=True).stack()\
    .to_frame('sent_str')
SENTS.index.names = OHCO[:2]
SENTS = SENTS[~SENTS['sent_str'].str.match(r'^\s*$')] # Remove empty paragraphs
SENTS.sent_str = SENTS.sent_str.str.strip() # CRUCIAL TO REMOVE BLANK TOKENS

In [28]:
token_pat = r"[\s',-]+"
TOKENS = SENTS['sent_str'].str.split(token_pat, expand=True).stack()\
    .to_frame('token_str')
TOKENS.index.names = OHCO[:5]
TOKENS['term_str'] = TOKENS.token_str.replace(r'[\W_]+', '', regex=True).str.lower()
VOCAB = TOKENS.term_str.value_counts().to_frame('n').reset_index().rename(columns={'index':'term_str'})
VOCAB.index.name = 'term_id'
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()



In [29]:
TOKENS = pd.concat([TOKENS], keys=[1], names=['chap_num'])

In [30]:
TOKENS.head()

token_str  term_str
chap_num para_num sent_num token_num                    
1        4        0        0          Produced  produced
                           1                by        by
                           2              Jack      jack
                           3            Voller    voller
         13       0        0               THE       the

In [31]:
VOCAB.head()

,term_str,n,p
term_id,,,
0,the,2637,0.046228
1,and,2079,0.036446
2,to,2068,0.036253
3,,1852,0.032467
4,of,1500,0.026296


In [32]:
baron_PARAS = PARAS
baron_SENTS = SENTS
baron_TOKENS = TOKENS
baron_VOCAB = VOCAB

In [33]:
baron_TOKENS.to_csv(f"{output_dir}/baron-TOKENS.csv")


## The Castle of Otranto

In [34]:
src_file = f"{data_home}/high-gothic/pg696.txt"
OHCO = ['chap_num', 'para_num', 'sent_num', 'token_num']

In [35]:
LINES = pd.DataFrame(open(src_file, 'r', encoding='utf-8-sig').readlines(), columns=['line_str'])
LINES.index.name = 'line_num'
LINES.line_str = LINES.line_str.str.replace(r'\n+', ' ', regex=True).str.strip()
title = LINES.loc[0].line_str.replace('The Project Gutenberg EBook of ', '')
clip_pats = [
    r"\*\*\*\s*START OF (?:THE|THIS) PROJECT",
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT"
]
pat_a = LINES.line_str.str.match(clip_pats[0])
pat_b = LINES.line_str.str.match(clip_pats[1])
line_a = LINES.loc[pat_a].index[0] + 1
line_b = LINES.loc[pat_b].index[0] - 1
LINES = LINES.loc[line_a : line_b]

chap_pat = r"^\s*(?:chapter)\s+[IVXLCDM]+"
chap_lines = LINES.line_str.str.match(chap_pat, case=False) # Returns a truth vector

LINES.loc[chap_lines, 'chap_num'] = [i+1 for i in range(LINES.loc[chap_lines].shape[0])]
LINES.chap_num = LINES.chap_num.ffill()

In [36]:
LINES = LINES.dropna(subset=['chap_num']) # Remove everything before Chapter 1
LINES = LINES.loc[~chap_lines] # Remove chapter heading lines; their work is done
LINES.chap_num = LINES.chap_num.astype('int') # Convert chap_num from float to int

In [37]:
CHAPS = LINES.groupby(OHCO[:1])\
    .line_str.apply(lambda x: '\n'.join(x))\
    .to_frame('chap_str')
CHAPS['chap_str'] = CHAPS.chap_str.str.strip()


In [38]:
CHAPS.head()

,chap_str
chap_num,
1,"Manfred, Prince of Otranto, had one son and on..."
2,"Matilda, who by Hippolita’s order had retired ..."
3,Manfred’s heart misgave him when he beheld the...
4,The sorrowful troop no sooner arrived at the c...
5,Every reflection which Manfred made on the Fri...


In [39]:
para_pat = r'\n\n+'
PARAS = CHAPS['chap_str'].str.split(para_pat, expand=True).stack()\
    .to_frame('para_str').sort_index()
PARAS.index.names = OHCO[:2]

In [40]:
PARAS['para_str'] = PARAS['para_str'].str.replace(r'\n', ' ', regex=True)
PARAS['para_str'] = PARAS['para_str'].str.strip()
PARAS = PARAS[~PARAS['para_str'].str.match(r'^\s*$')] # Remove empty paragraphs

In [41]:
sent_pat = r'[.?!;:]+'
SENTS = PARAS['para_str'].str.split(sent_pat, expand=True).stack()\
    .to_frame('sent_str')
SENTS.index.names = OHCO[:3]
SENTS = SENTS[~SENTS['sent_str'].str.match(r'^\s*$')] # Remove empty paragraphs
SENTS.sent_str = SENTS.sent_str.str.strip() # CRUCIAL TO REMOVE BLANK TOKENS

In [42]:
token_pat = r"[\s',-]+"
TOKENS = SENTS['sent_str'].str.split(token_pat, expand=True).stack()\
    .to_frame('token_str')
TOKENS.index.names = OHCO[:4]
TOKENS
TOKENS['term_str'] = TOKENS.token_str.replace(r'[\W_]+', '', regex=True).str.lower()
VOCAB = TOKENS.term_str.value_counts().to_frame('n').reset_index().rename(columns={'index':'term_str'})
VOCAB.index.name = 'term_id'
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()



In [43]:
TOKENS.head()

token_str term_str
chap_num para_num sent_num token_num                   
1        0        0        0           Manfred  manfred
                           1            Prince   prince
                           2                of       of
                           3           Otranto  otranto
                           4               had      had

In [44]:
VOCAB.head()

,term_str,n,p
term_id,,,
0,the,1930,0.054160
1,to,1216,0.034124
2,,981,0.027529
3,of,961,0.026968
4,and,798,0.022394


In [45]:
otranto_CHAPS = CHAPS
otranto_PARAS = PARAS
otranto_SENTS = SENTS
otranto_TOKENS = TOKENS
otranto_VOCAB = VOCAB

In [46]:
otranto_TOKENS.to_csv(f"{output_dir}/otranto-TOKENS.csv")


## Document Lengths for High Gothic Novels

In [47]:
print("Udolpho len: ", len(udolpho_TOKENS))
print("BAron len: ", len(baron_TOKENS))
print("Otranto len: ", len(otranto_TOKENS))
high_gothic_lens = [len(udolpho_TOKENS), len(baron_TOKENS), len(otranto_TOKENS)]
print("Average High GOthic len: ", (sum(high_gothic_lens)/3))

Udolpho len:  71201
BAron len:  57043
Otranto len:  35635
Average High GOthic len:  54626.333333333336


# Classic/Romantic Gothic

## Melmoth the Wanderer

In [48]:
src_file = f"{data_home}/melmoth/pg53685.txt"
OHCO = ['chap_num', 'para_num', 'sent_num', 'token_num']

In [49]:
LINES = pd.DataFrame(open(src_file, 'r', encoding='utf-8-sig').readlines(), columns=['line_str'])
LINES.index.name = 'line_num'
LINES.line_str = LINES.line_str.str.replace(r'\n+', ' ', regex=True).str.strip()
title = LINES.loc[0].line_str.replace('The Project Gutenberg EBook of ', '')
clip_pats = [
    r"\*\*\*\s*START OF (?:THE|THIS) PROJECT",
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT"
]
pat_a = LINES.line_str.str.match(clip_pats[0])
pat_b = LINES.line_str.str.match(clip_pats[1])
line_a = LINES.loc[pat_a].index[0] + 1
line_b = LINES.loc[pat_b].index[0] - 1
LINES = LINES.loc[line_a : line_b]

chap_pat = r"^\s*(?:chapter)\s+[IVXLCDM]+"
chap_lines = LINES.line_str.str.match(chap_pat, case=False) # Returns a truth vector

LINES.loc[chap_lines, 'chap_num'] = [i+1 for i in range(LINES.loc[chap_lines].shape[0])]
LINES.chap_num = LINES.chap_num.ffill()

In [50]:
LINES = LINES.dropna(subset=['chap_num']) # Remove everything before Chapter 1
LINES = LINES.loc[~chap_lines] # Remove chapter heading lines; their work is done
LINES.chap_num = LINES.chap_num.astype('int') # Convert chap_num from float to int

In [51]:
CHAPS = LINES.groupby(OHCO[:1])\
    .line_str.apply(lambda x: '\n'.join(x))\
    .to_frame('chap_str')
CHAPS['chap_str'] = CHAPS.chap_str.str.strip()


In [52]:
para_pat = r'\n\n+'
PARAS = CHAPS['chap_str'].str.split(para_pat, expand=True).stack()\
    .to_frame('para_str').sort_index()
PARAS.index.names = OHCO[:2]

In [53]:
PARAS['para_str'] = PARAS['para_str'].str.replace(r'\n', ' ', regex=True)
PARAS['para_str'] = PARAS['para_str'].str.strip()
PARAS = PARAS[~PARAS['para_str'].str.match(r'^\s*$')] # Remove empty paragraphs

In [54]:
sent_pat = r'[.?!;:]+'
SENTS = PARAS['para_str'].str.split(sent_pat, expand=True).stack()\
    .to_frame('sent_str')
SENTS.index.names = OHCO[:3]
SENTS = SENTS[~SENTS['sent_str'].str.match(r'^\s*$')] # Remove empty paragraphs
SENTS.sent_str = SENTS.sent_str.str.strip() # CRUCIAL TO REMOVE BLANK TOKENS

In [55]:
token_pat = r"[\s',-]+"
TOKENS = SENTS['sent_str'].str.split(token_pat, expand=True).stack()\
    .to_frame('token_str')
TOKENS.index.names = OHCO[:4]
TOKENS
TOKENS['term_str'] = TOKENS.token_str.replace(r'[\W_]+', '', regex=True).str.lower()
VOCAB = TOKENS.term_str.value_counts().to_frame('n').reset_index().rename(columns={'index':'term_str'})
VOCAB.index.name = 'term_id'
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()



In [56]:
TOKENS.head()

token_str term_str
chap_num para_num sent_num token_num                   
1        0        0        0             Alive    alive
                           1             again    again
                  1        0              Then     then
                           1              show     show
                           2                me       me

In [57]:
VOCAB.head()

,term_str,n,p
term_id,,,
0,the,3697,0.063471
1,of,2108,0.036191
2,and,1870,0.032105
3,to,1625,0.027898
4,,1309,0.022473


In [58]:
melmoth_CHAPS = CHAPS
melmoth_PARAS = PARAS
melmoth_SENTS = SENTS
melmoth_TOKENS = TOKENS
melmoth_VOCAB = VOCAB

In [59]:
print("Melmoth len: ", len(melmoth_TOKENS))

Melmoth len:  58247


In [60]:
melmoth_TOKENS.to_csv(f"{output_dir}/melmoth-TOKENS.csv")


In [62]:
high_goth_corp = [udolpho_TOKENS, baron_TOKENS, otranto_TOKENS]
ids = [3268, 5182, 696]
HG_CORPUS = pd.concat(high_goth_corp, keys=ids, names=['doc_id'])
HG_CORPUS.to_csv(f"{output_dir}/hg-corpus.csv")

In [63]:
HG_CORPUS

token_str    term_str
doc_id chap_num para_num sent_num token_num                        
3268   1        0        0        0                home        home
                                  1                  is          is
                                  2                 the         the
                                  3              resort      resort
                                  4                  Of          of
...                                                 ...         ...
696    5        121      5        40              taken       taken
                                  41         possession  possession
                                  42                 of          of
                                  43                his         his
                                  44               soul        soul

[163879 rows x 2 columns]